# Fictional Context Story – Public Health NGO Dataset

### Context
In 2024, the international NGO **Global Health Reach (GHR)** launched a city-wide initiative to monitor and improve primary healthcare services across two hospitals and three clinics located in different neighborhoods of the fictional bilingual West-African city of **Kambara City**. The city is officially bilingual (French and English), but linguistic practices vary widely by neighborhood.

### The Clinics
Each of the five neighborhood centers operates semi-independently, using its own staff, tools, workflows, and data entry practices. This decentralization creates significant inconsistencies and quality issues across the datasets, which is why GHR recruited a skilled data analyst to lead the consolidation and cleaning effort.

### Francophone neighborhoods (3 clinics)
- **Clinique Saint-Bernard (CSB)** – located in a semi-urban area, with older computer systems and paper-based archives.
- **Centre Médical Luko (CML)** – situated in a densely populated neighborhood; nurses often enter data in French and occasionally mix in local dialect terms.
- **Hôpital de Kanza (HKZ)** – the largest francophone hospital; recently digitized, but migration caused inconsistent data formats.

### Anglophone neighborhoods (2 clinics)
- **Abeni Health Center (LAHC)** – modern EMR system in use, but frequent software updates cause CSV export inconsistencies.
- **West Karumo Medical Post (WKMP)** – located in a remote part of the city, sometimes offline, resulting in delayed or partially missing records. Data is first recorded manually and then typed by volunteers with varying accuracy.

### Longitudinal Data
Patients are tracked across multiple visits throughout the year, allowing GHR to monitor individual health trends over time. Each patient may have 1–5 visits, with measurements such as weight, height, temperature, blood pressure, diagnosis, and vaccination status recorded for each visit. Notes may include logistical observations or qualitative feedback about patient satisfaction.

### Objective
GHR’s program manager mandated a comprehensive data quality assessment to clean, reconcile, and standardize the information from all five facilities. The mission involves identifying inconsistencies, harmonizing formats and definitions, resolving missing or contradictory records, handling outliers, and producing a consolidated dataset robust enough to support evidence-based decision-making and donor-required reporting in English. This effort is intended to lay the groundwork for stronger data governance, reliable longitudinal health monitoring, and actionable insights across all neighborhoods of Kambara City.

### Scope of This Notebook (Part 2)

This notebook focuses exclusively on **Part 2 of the data cleaning pipeline**, which includes:  

- **Cleaning and harmonization of patient-level and clinic/hospital-level columns**  
- **Merging the five datasets into a consolidated dataset**  
- **Converting columns to appropriate data types**  

After completing this notebook, the dataset will be ready for **Notebook 3**, where we will handle **missing value imputation, and further data transformations**.

**Author:** J-F Jutras  
**Date:** December 2025  
**Dataset:** GHR-Dataset (5 CSV files)

## 2.1-Data Loading and Overview

In [1]:
import os
import pandas as pd
import warnings

#Ignore RuntimeWarnings from pandas formatting when printing DataFrames.
#These warnings are usually caused by NaN or mixed-type columns being interpreted as numeric.
warnings.simplefilter(action='ignore', category=RuntimeWarning)

#Folder where the dataset is mounted by Kaggle
dataset_folder = "/kaggle/input/ghr-dataset-part1"

#List all files in the dataset folder
print("Files in the dataset folder:")
print(os.listdir(dataset_folder))

#Load all CSVs into a dictionary of DataFrames
dfs = {}
for file in os.listdir(dataset_folder):
    if file.endswith(".csv"):
        path = os.path.join(dataset_folder, file)
        dfs[file] = pd.read_csv(path, encoding = 'utf-8', on_bad_lines = 'skip')

#Quick check
for name, df in dfs.items():
    print(f"{name} shape: {df.shape}")
    print(f"{name} head: {df.head(2)}")

Files in the dataset folder:
['Donnees_Kanza_Kambara_cleaned.csv', 'Donnees_Abeni_Kambara_cleaned.csv', 'Donnees_WKMP_Kambara_cleaned.csv', 'Donnees_Luko_Kambara_cleaned.csv', 'Donnees_CSB_Kambara_cleaned.csv']
Donnees_Kanza_Kambara_cleaned.csv shape: (100, 22)
Donnees_Kanza_Kambara_cleaned.csv head:   id_patient  visit_number  visit_date       clinic_name patient_language  \
0       H101             1  2024-03-01  Hôpital de Kanza           French   
1       H101             2  2024-03-10  Hôpital de Kanza           French   

  first_name last_name sex  birth_date  age  ... temperature blood_pressure  \
0    Aminata    DIALLO   F  1990-01-01   34  ...      39.5 C         120/80   
1    aminata    diallo   f  01/01/1990   34  ...        37.0         118/78   

  pulse oxygen_saturation  diagnosis vaccination_status  \
0    95               94%  Méningite            Complet   
1    80               98%  Méningite            complet   

  hospitalization_required hospitalization_days bl

| Nom de colonne (FR) | Column Name (EN) | Description | Scope |
|--------------------|-----------------|-------------|-------|
| id_patient | id_patient | Unique patient identifier | Clinic/Hospital |
| num_visite | visit_number | Visit number for the patient (1–5) | Clinic/Hospital |
| date_visite | visit_date | Date of the visit | Clinic/Hospital |
| nom_clinique | clinic_name | Name of the clinic or hospital where the visit occurred | Clinic/Hospital |
| langue_patient | patient_language | Patient's preferred language (French/English) | Clinic/Hospital |
| prenom | first_name | Patient's first name | Clinic/Hospital |
| nom | last_name | Patient's last name | Clinic/Hospital |
| sexe | sex | Patient's sex | Clinic/Hospital |
| date_naissance | birth_date | Patient's date of birth | Clinic/Hospital |
| age | age | Patient's age at the time of the visit | Clinic/Hospital |
| taille | height | Patient's height | Clinic/Hospital |
| poids | weight | Patient's weight | Clinic/Hospital |
| temperature | temperature | Body temperature measured during the visit | Clinic/Hospital |
| pression_sanguine | blood_pressure | Blood pressure measurement | Clinic/Hospital |
| pouls | pulse | Pulse rate (beats per minute) | Clinic/Hospital |
| saturation_oxygene | oxygen_saturation | Blood oxygen saturation | Clinic/Hospital |
| diagnostic | diagnosis | Diagnosis given during the visit. Possible values: Malaria, Tuberculosis, Meningitis, Trypanosomiasis, Cholera, Typhoid, Dengue | Clinic/Hospital |
| statut_vaccinal | vaccination_status | Vaccination status. Possible values: Complete, Partial, Not vaccinated | Clinic/Hospital |
| hospitalisation_necessaire | hospitalization_required | Whether hospitalization was required. Possible values: Y, N | Clinic/Hospital |
| jours_hospitalisation | hospitalization_days | Number of days hospitalized if required | Hospital only |
| analyses_sanguines_necessaires | blood_tests_required | Whether blood tests were required. Possible values: Y, N | Hospital only |
| imagerie_necessaire | imaging_required | Whether imaging exams (X-ray, ultrasound, etc.) were required. Possible values: Y, N | Hospital only |


## 2.2-Raw Column Cleaning and Harmonization - Patient-Related Columns

In [2]:
#Define patient-related columns
patient_cols = [
    'first_name', 'last_name', 'sex', 'birth_date', 'age', 'height', 'weight', 'temperature',
    'blood_pressure', 'pulse', 'oxygen_saturation', 'diagnosis', 'vaccination_status'
]

#Display first 20 rows of names
for file_name, df in dfs.items():
    print(f"=== File: {file_name} ===")
    print("First 20 first_name values:")
    print(df['first_name'].head(20).tolist())
    print("First 20 last_name values:")
    print(df['last_name'].head(20).tolist())
    print("\n")

=== File: Donnees_Kanza_Kambara_cleaned.csv ===
First 20 first_name values:
['Aminata', 'aminata', 'Bakary', 'bakary', 'Ndeye', 'ndeye', 'Moussa', 'moussa', 'Aisha', 'aisha', 'Cheikh', 'cheikh', 'Fatou', 'fatou', 'Ibrahim', 'ibrahim', 'Coumba', 'coumba', 'Mamadou', 'Mamadou Ba']
First 20 last_name values:
['DIALLO', 'diallo', 'SOW', 'sow', 'FALL', 'fall', 'KANE', 'kane', 'BA', 'ba', 'DIOP', 'diop', 'NDIAYE', 'ndiaye', 'DIARRA', 'diarra', 'SOW', 'sow', 'BA', nan]


=== File: Donnees_Abeni_Kambara_cleaned.csv ===
First 20 first_name values:
['Chinelo', 'Chinelo', 'Chinelo', 'Kwasi', 'Kwasi', 'Aisha', 'Aisha', 'Tunde', 'TUNDE', 'Tunde', 'Zuri', 'Zuri', 'Musa', 'Ngozi', 'Ngozi', 'Kofi', 'Amara', 'Amara', 'Jengo', 'Lebo']
First 20 last_name values:
['Okoro', 'Okoro', 'Okoro', 'Adom', 'Adom', 'Diallo', 'Diallo', 'Olatunji', 'OLATUNJI', 'Olatunji', 'Kamau', 'Kamau', 'Sanda', 'Eze', 'Eze', 'Mensah', 'Uzoma', 'Uzoma', 'Mkali', 'Dube']


=== File: Donnees_WKMP_Kambara_cleaned.csv ===
First 20 fi

After inspecting the first 20 values for `first_name` and `last_name` in all five CSV files, the following issues were identified:

**1. Inconsistent capitalization**
- Names appear in different cases.

**2. Full names in `first_name`**
- Some `first_name` entries contain both first name and last name.

**3. Duplicated information between `first_name` and `last_name`**
- The last name is sometimes repeated in both columns.
- Note: If multiple visits show the same format consistently, it might be the correct form. Otherwise, validation against reliable data sources is recommended.

**4. Missing values / inconsistencies**

**5. Noms composés and hyphenated names**
- First names may contain hyphens (e.g., `'Jean-Luc'`) or multiple parts, requiring careful capitalization.

In [3]:
#Function to clean names
def clean_names(df, first_col='first_name', last_col='last_name'):
    #Strip whitespace and standardize missing values
    df[first_col] = df[first_col].astype(str).str.strip().replace(['nan', 'None', '', 'NaN', '<na>'], pd.NA)
    df[last_col] = df[last_col].astype(str).str.strip().replace(['nan', 'None', '', 'NaN', '<na>'], pd.NA)

    #Proper capitalization, preserving hyphens
    def proper_case(name):
        if pd.isna(name):
            return pd.NA
        return '-'.join([part.capitalize() for part in name.split('-')])

    df[first_col] = df[first_col].apply(proper_case)
    df[last_col] = df[last_col].apply(proper_case)

    #Split full first names into first and last if needed
    split_names = df[first_col].str.split(' ', n = 1, expand = True)
    df['first_name_clean'] = split_names[0]
    df['last_extracted'] = split_names[1] if split_names.shape[1] > 1 else pd.NA

    #Decide which last name to keep
    df['last_name_clean'] = df.apply(
        lambda row: row[last_col] if pd.notna(row[last_col]) and str(row[last_col]).strip() != str(row['last_extracted']).strip()
        else row['last_extracted'],
        axis=1
    )

    #Replace any residual '<na>' or empty strings with pd.NA
    df['last_name_clean'] = df['last_name_clean'].replace(['<na>', '', 'nan', 'None', 'NaN'], pd.NA)

    #Proper capitalization again after cleaning
    df['last_name_clean'] = df['last_name_clean'].apply(proper_case)

    #Drop temporary column
    df.drop(columns=['last_extracted'], inplace=True)

    return df

#Apply cleaning and replace original columns for all datasets
for file_name, df in dfs.items():
    #Clean names
    df = clean_names(df)
    
    #Replace original columns
    df.drop(columns=['first_name', 'last_name'], inplace = True)
    df.rename(columns={'first_name_clean':'first_name', 'last_name_clean':'last_name'}, inplace = True)
    
    #Save back in dictionary
    dfs[file_name] = df

#Quick verification
for file_name, df in dfs.items():
    print(f"=== File: {file_name} ===")
    print("First 20 first_name values:")
    print(df['first_name'].head(20).tolist())
    print("First 20 last_name values:")
    print(df['last_name'].head(20).tolist())
    print("\n")

=== File: Donnees_Kanza_Kambara_cleaned.csv ===
First 20 first_name values:
['Aminata', 'Aminata', 'Bakary', 'Bakary', 'Ndeye', 'Ndeye', 'Moussa', 'Moussa', 'Aisha', 'Aisha', 'Cheikh', 'Cheikh', 'Fatou', 'Fatou', 'Ibrahim', 'Ibrahim', 'Coumba', 'Coumba', 'Mamadou', 'Mamadou']
First 20 last_name values:
['Diallo', 'Diallo', 'Sow', 'Sow', 'Fall', 'Fall', 'Kane', 'Kane', 'Ba', 'Ba', 'Diop', 'Diop', 'Ndiaye', 'Ndiaye', 'Diarra', 'Diarra', 'Sow', 'Sow', 'Ba', 'Ba']


=== File: Donnees_Abeni_Kambara_cleaned.csv ===
First 20 first_name values:
['Chinelo', 'Chinelo', 'Chinelo', 'Kwasi', 'Kwasi', 'Aisha', 'Aisha', 'Tunde', 'Tunde', 'Tunde', 'Zuri', 'Zuri', 'Musa', 'Ngozi', 'Ngozi', 'Kofi', 'Amara', 'Amara', 'Jengo', 'Lebo']
First 20 last_name values:
['Okoro', 'Okoro', 'Okoro', 'Adom', 'Adom', 'Diallo', 'Diallo', 'Olatunji', 'Olatunji', 'Olatunji', 'Kamau', 'Kamau', 'Sanda', 'Eze', 'Eze', 'Mensah', 'Uzoma', 'Uzoma', 'Mkali', 'Dube']


=== File: Donnees_WKMP_Kambara_cleaned.csv ===
First 20 firs

We will now process the `birth_date` column in each dataset. The cleaning and standardization of dates follow **the same approach used for `visit_date` in Notebook 1**

In [4]:
import re
import numpy as np

#We are only investigating existing date patterns (NOT cleaning yet)
#This helps us discover how dates are actually written in all datasets.

date_cols = ['birth_date']

def extract_pattern(date_str):
    pattern = ""
    for c in str(date_str):
        if c.isdigit():
            pattern += "d"
        elif c.isalpha():
            pattern += "a"
        else:
            pattern += c
    return pattern

for name, df in dfs.items():
    print(f"\n=== Dataset: {name} ===")

    for col in date_cols:
        if col not in df.columns:
            continue
    
        print(f"\nColumn: {col}")

        #Unique non-null values
        unique_dates = df[col].dropna().astype(str).unique()

        #Extract structural patterns
        patterns_found = set()
        for val in unique_dates:
            pattern = extract_pattern(val)
            patterns_found.add(pattern)

        #Display discovered patterns only
        print("Detected date patterns:")
        for pattern in patterns_found:
            print(f"  {pattern}")


=== Dataset: Donnees_Kanza_Kambara_cleaned.csv ===

Column: birth_date
Detected date patterns:
  dd/dd/dddd
  dddd-dd-dd

=== Dataset: Donnees_Abeni_Kambara_cleaned.csv ===

Column: birth_date
Detected date patterns:
  dddd-dd-dd

=== Dataset: Donnees_WKMP_Kambara_cleaned.csv ===

Column: birth_date
Detected date patterns:
  dd-dd-dddd
  dd/dd/dddd
  dddd/dd/dd
  dddd-dd-dd

=== Dataset: Donnees_Luko_Kambara_cleaned.csv ===

Column: birth_date
Detected date patterns:
  dd/dd/dddd
  dddd-dd-dd

=== Dataset: Donnees_CSB_Kambara_cleaned.csv ===

Column: birth_date
Detected date patterns:
  dd/dd/dddd
  dd-dd-dddd
  dddd/dd/dd
  dddd-dd-dd


After inspecting the detected date patterns in the `birth_date` column across all datasets, we observe multiple formats. For now, we will **repeat the cleaning steps from Notebook 1** on `birth_date`, as the centralized function repository has not yet been implemented. In the future, centralizing reusable functions would improve efficiency, consistency, and maintainability, but for the current workflow we proceed by reapplying the existing logic.

In [5]:
def clean_dates(series):
  
    def standardize(val):
        if pd.isna(val):
            return pd.NA

        #Convert to string and strip spaces
        val = str(val).strip()

        #Normalize separators to "-"
        val = re.sub(r"[./]", "-", val)

        #Reorder DD-MM-YYYY → YYYY-MM-DD
        m = re.match(r"^(?P<d>\d{1,2})-(?P<m>\d{1,2})-(?P<y>\d{4})$", val)
        if m:
            val = f"{m.group('y')}-{m.group('m')}-{m.group('d')}"

        #Normalize YYYY/MM/DD → YYYY-MM-DD
        val = re.sub(r"^(\d{4})-(\d{1,2})-(\d{1,2})$", r"\1-\2-\3", val)

        return val

    return series.apply(standardize)

#Apply to all datasets
for name, df in dfs.items():
    if "birth_date" in df.columns:
        df["birth_date"] = clean_dates(df["birth_date"])

In [6]:
#Define categorical columns to inspect
categorical_cols = ['sex', 'diagnosis', 'vaccination_status']

#Loop through datasets and inspect unique values for categorical patient-related columns
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    for col in categorical_cols:
        if col in df.columns:
            unique_vals = df[col].dropna().unique()
            print(f"Column: {col} (unique values count : {len(unique_vals)})")


Dataset: Donnees_Kanza_Kambara_cleaned.csv
Column: sex (unique values count : 4)
Column: diagnosis (unique values count : 9)
Column: vaccination_status (unique values count : 6)

Dataset: Donnees_Abeni_Kambara_cleaned.csv
Column: sex (unique values count : 2)
Column: diagnosis (unique values count : 7)
Column: vaccination_status (unique values count : 6)

Dataset: Donnees_WKMP_Kambara_cleaned.csv
Column: sex (unique values count : 4)
Column: diagnosis (unique values count : 8)
Column: vaccination_status (unique values count : 6)

Dataset: Donnees_Luko_Kambara_cleaned.csv
Column: sex (unique values count : 4)
Column: diagnosis (unique values count : 11)
Column: vaccination_status (unique values count : 6)

Dataset: Donnees_CSB_Kambara_cleaned.csv
Column: sex (unique values count : 4)
Column: diagnosis (unique values count : 8)
Column: vaccination_status (unique values count : 8)


In [7]:
#Inspect unique values for sex, diagnosis and vaccination_status
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    if 'sex' in df.columns:
        print("Sex:", df['sex'].unique())
    if 'diagnosis' in df.columns:
        print("Diagnosis:", df['diagnosis'].unique())
    if 'vaccination_status' in df.columns:
        print("Vaccination Status:", df['vaccination_status'].unique())


Dataset: Donnees_Kanza_Kambara_cleaned.csv
Sex: ['F' 'f' 'M' 'm']
Diagnosis: ['Méningite' 'Tuberculose' 'Choléra' 'Cholera' 'Paludisme' 'Malaria'
 'Homa' 'Dengue' 'Trypanosomiase']
Vaccination Status: ['Complet' 'complet' 'Partiel' 'partiel' 'Non vacciné' 'Non vacc']

Dataset: Donnees_Abeni_Kambara_cleaned.csv
Sex: ['F' 'M']
Diagnosis: ['Malaria' 'Typhoid' 'Cholera' 'Dengue' nan 'Tuberculosis' 'Meningitis'
 'Trypanosomiasis']
Vaccination Status: ['Fully vaccinated' 'Partially vaccinated' 'Not vaccinated'
 'Fully Vaccinated' 'Partially Vaccinated' 'fully vaccinated']

Dataset: Donnees_WKMP_Kambara_cleaned.csv
Sex: ['M' 'm' 'F' 'f']
Diagnosis: ['Malaria' 'malaria' 'Typhoid' 'Trypanosomiasis' 'Dengue' 'Cholera'
 'Tuberculosis' 'Meningitis' nan]
Vaccination Status: ['Complete' 'Partial' 'Not vacc' 'Not vaccinated' 'complete' 'partial']

Dataset: Donnees_Luko_Kambara_cleaned.csv
Sex: ['M' 'm' 'F' 'f']
Diagnosis: ['Paludisme' 'Malaria' 'Trypanosomiase' 'Homa' 'Typhoïde' 'Méningite'
 'Dengue

In [8]:
#Function to clean categorical columns
def clean_categoricals(df):
    #Helper to normalize missing values
    def normalize_missing(x):
        if pd.isna(x):
            return pd.NA
        x = str(x).strip().lower()
        if x in ['nan', 'none', '<na>', 'na']:
            return pd.NA
        return x
    
    #Clean 'sex' column
    if 'sex' in df.columns:
        df['sex'] = df['sex'].apply(normalize_missing)
        df['sex'] = df['sex'].replace({'f': 'F', 'm': 'M'})
    
    #Clean 'diagnosis' column
    if 'diagnosis' in df.columns:
        df['diagnosis'] = df['diagnosis'].apply(normalize_missing)
        df['diagnosis'] = df['diagnosis'].replace({
            #French → English
            'paludisme': 'Malaria',
            'méningite': 'Meningitis',
            'tuberculose': 'Tuberculosis',
            'choléra': 'Cholera',
            'typhoïde': 'Typhoid',
            'typhoide': 'Typhoid',
            'trypanosomiase': 'Trypanosomiasis',
            #English normalization
            'malaria': 'Malaria',
            'tuberculosis': 'Tuberculosis',
            'meningitis': 'Meningitis',
            'cholera': 'Cholera',
            'dengue': 'Dengue',
            'typhoid': 'Typhoid',
            'trypanosomiasis': 'Trypanosomiasis',
            'homa': 'Malaria'  #Homa is Swahili for Malaria
        })
    
    #Clean 'vaccination_status' column
    if 'vaccination_status' in df.columns:
        df['vaccination_status'] = df['vaccination_status'].apply(normalize_missing)
        df['vaccination_status'] = df['vaccination_status'].replace({
            'complet': 'Complete',
            'complete': 'Complete',
            'complét': 'Complete',
            'partiel': 'Partial',
            'partial': 'Partial',
            'partielle': 'Partial',
            'fully vaccinated': 'Complete',
            'partially vaccinated': 'Partial',
            'not vaccinated': 'Not vaccinated',
            'non vacciné': 'Not vaccinated',
            'non vacc': 'Not vaccinated',
            'not vacc': 'Not vaccinated',
            'non vaciné': 'Not vaccinated'
        })
    
    return df

#Apply to all datasets
for name, df in dfs.items():
    dfs[name] = clean_categoricals(df)

#Quick check of cleaned categorical columns
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    if 'sex' in df.columns:
        print("Sex:", df['sex'].unique())
    if 'diagnosis' in df.columns:
        print("Diagnosis:", df['diagnosis'].unique())
    if 'vaccination_status' in df.columns:
        print("Vaccination Status:", df['vaccination_status'].unique())


Dataset: Donnees_Kanza_Kambara_cleaned.csv
Sex: ['F' 'M']
Diagnosis: ['Meningitis' 'Tuberculosis' 'Cholera' 'Malaria' 'Dengue'
 'Trypanosomiasis']
Vaccination Status: ['Complete' 'Partial' 'Not vaccinated']

Dataset: Donnees_Abeni_Kambara_cleaned.csv
Sex: ['F' 'M']
Diagnosis: ['Malaria' 'Typhoid' 'Cholera' 'Dengue' <NA> 'Tuberculosis' 'Meningitis'
 'Trypanosomiasis']
Vaccination Status: ['Complete' 'Partial' 'Not vaccinated']

Dataset: Donnees_WKMP_Kambara_cleaned.csv
Sex: ['M' 'F']
Diagnosis: ['Malaria' 'Typhoid' 'Trypanosomiasis' 'Dengue' 'Cholera' 'Tuberculosis'
 'Meningitis' <NA>]
Vaccination Status: ['Complete' 'Partial' 'Not vaccinated']

Dataset: Donnees_Luko_Kambara_cleaned.csv
Sex: ['M' 'F']
Diagnosis: ['Malaria' 'Trypanosomiasis' 'Typhoid' 'Meningitis' 'Dengue' 'Cholera'
 'Tuberculosis']
Vaccination Status: ['Partial' 'Not vaccinated' 'Complete']

Dataset: Donnees_CSB_Kambara_cleaned.csv
Sex: ['F' 'M']
Diagnosis: ['Malaria' 'Dengue' 'Trypanosomiasis' 'Typhoid' 'Cholera' 'Tub

In [9]:
#Starting to handle numeric-like columns (kept as strings for now)
for name, df in dfs.items():
    if 'blood_pressure' in df.columns:
        #Split 'blood_pressure' into systolic and diastolic without converting to numbers
        bp_split = df['blood_pressure'].astype(str).str.extract(r'(?P<systolic>[^/-]+)[/-](?P<diastolic>[^/-]+)')
        
        #Assign to new columns (still as strings)
        df['blood_pressure_systolic'] = bp_split['systolic']
        df['blood_pressure_diastolic'] = bp_split['diastolic']
        
        #Save back in dictionary
        dfs[name] = df

#Quick verification
for name, df in dfs.items():
    if 'blood_pressure_systolic' in df.columns:
        print(f"\nDataset: {name}")
        print(df[['blood_pressure_systolic', 'blood_pressure_diastolic']].head(10))


Dataset: Donnees_Kanza_Kambara_cleaned.csv
  blood_pressure_systolic blood_pressure_diastolic
0                     120                       80
1                     118                       78
2                     130                       90
3                     128                       88
4                     110                       70
5                     108                       68
6                     145                       95
7                     140                       90
8                     125                       85
9                     120                       80

Dataset: Donnees_Abeni_Kambara_cleaned.csv
  blood_pressure_systolic blood_pressure_diastolic
0                     120                       80
1                     118                       78
2                     135                       90
3                     110                       70
4                     108                       68
5                     125                    

In [10]:
# Function to identify values that contain non-numeric characters
def find_non_numeric(series):
    # Convert all values to string
    s = series.astype(str)
    
    # Mask values that contain any character other than digits or dot
    mask = s.apply(lambda x: bool(re.search(r"[^\d.]", x)))
    
    # Return unique non-numeric values
    return s[mask].unique()

# List of numeric-like columns to inspect (kept as strings for now)
numeric_cols = ['age', 'height', 'weight', 'temperature', 
                'blood_pressure_systolic', 'blood_pressure_diastolic', 'pulse', 'oxygen_saturation']

# Check all datasets
for name, df in dfs.items():
    print(f"\n=== Dataset: {name} ===")
    for col in numeric_cols:
        if col in df.columns:
            non_numeric_values = find_non_numeric(df[col])
            print(f"Non-numeric values in '{col}': {non_numeric_values}")


=== Dataset: Donnees_Kanza_Kambara_cleaned.csv ===
Non-numeric values in 'age': []
Non-numeric values in 'height': ['165 cm' '178 cm' '160cm' '170 cm' '135 cm' '155 cm' '180 cm' '163 cm'
 '175 cm' '168 cm' '162 cm' '160 cm' '172 cm']
Non-numeric values in 'weight': ['65 kg' '80 kg' '60kg' '75 kg' '68 kg' '32 kg' '70 kg' '78 kg' '62 kg'
 '85 kg' '60 kg' '88 kg' '55 kg']
Non-numeric values in 'temperature': ['39.5 C' '39.8 C']
Non-numeric values in 'blood_pressure_systolic': []
Non-numeric values in 'blood_pressure_diastolic': []
Non-numeric values in 'pulse': []
Non-numeric values in 'oxygen_saturation': ['94%' '98%' '96%' '97%' '90%' '92%' '93%' '95%']

=== Dataset: Donnees_Abeni_Kambara_cleaned.csv ===
Non-numeric values in 'age': []
Non-numeric values in 'height': ['165.5 cm' '165 cm' '5\'5"' '150.2 cm' '151 cm' '5\'2"' '158 cm' '178 cm'
 '5\'10"' '160 cm' '170 cm' '168 cm' '3\'0"' '162 cm' '175 cm' '130 cm'
 '5\'6"' '163 cm' '5\'7"' '180 cm' '166 cm' '173 cm' '155 cm' '5\'1"'
 '176

In [11]:
def clean_numeric_column(series, col_name=None):
    def convert_value(value):
        if pd.isna(value):
            return np.nan
        val_str = str(value).strip()

        #Height conversions
        if col_name == 'height':
            #Feet and inches
            feet_inch = re.match(r"(\d+)'(\d+)", val_str)
            if feet_inch:
                feet, inch = map(int, feet_inch.groups())
                return float(feet*30.48 + inch*2.54)
            #Meters
            meter_match = re.match(r"(\d+\.?\d*)\s*m", val_str)
            if meter_match:
                meters = float(meter_match.group(1))
                return meters * 100
            #Remove 'cm' or other characters
            val_str = re.sub(r"[^\d.]", "", val_str)
            return float(val_str) if val_str else np.nan

        #Weight conversions
        elif col_name == 'weight':
            #lbs to kg
            lbs_match = re.match(r"(\d+\.?\d*)\s*lbs", val_str, re.IGNORECASE)
            if lbs_match:
                lbs = float(lbs_match.group(1))
                return round(lbs * 0.453592, 2)
            #Remove 'kg' or other characters
            val_str = re.sub(r"[^\d.]", "", val_str)
            return float(val_str) if val_str else np.nan

        #Other numeric columns
        else:
            val_str = re.sub(r"[^\d.]", "", val_str)
            return float(val_str) if val_str else np.nan

    return series.apply(convert_value)

#Apply numeric cleaning to all datasets
for file_name, df in dfs.items():
    for col in numeric_cols:
        if col in df.columns:
            df[col] = clean_numeric_column(df[col], col_name=col)
    dfs[file_name] = df

#Quick verification: show first 10 values
for file_name, df in dfs.items():
    print(f"=== File: {file_name} ===")
    for col in numeric_cols:
        print(f"{col} first 10 values: {df[col].head(20).tolist()}")
    print("\n")

=== File: Donnees_Kanza_Kambara_cleaned.csv ===
age first 10 values: [34.0, 34.0, 48.0, 48.0, 23.0, 23.0, 64.0, 64.0, 38.0, 38.0, 9.0, 9.0, 53.0, 53.0, 32.0, 32.0, 43.0, 43.0, 69.0, 69.0]
height first 10 values: [165.0, 165.0, 178.0, 178.0, 160.0, 160.0, 170.0, 170.0, 170.0, 170.0, 135.0, 135.0, 155.0, 155.0, 180.0, 180.0, 163.0, 163.0, 175.0, 175.0]
weight first 10 values: [65.0, 64.5, 80.0, 79.0, 60.0, 59.0, 75.0, 75.0, 68.0, 67.5, 32.0, 32.0, 70.0, 69.5, 78.0, 78.0, 62.0, 62.0, 85.0, 84.5]
temperature first 10 values: [39.5, 37.0, 38.2, 37.1, 40.1, 37.5, 37.5, 37.0, 39.8, 37.2, 39.0, 37.5, 38.8, 37.0, 39.1, 37.0, 37.0, 37.1, 39.5, 37.5]
blood_pressure_systolic first 10 values: [120.0, 118.0, 130.0, 128.0, 110.0, 108.0, 145.0, 140.0, 125.0, 120.0, 105.0, 100.0, 140.0, 135.0, 130.0, 125.0, 120.0, 120.0, 150.0, 140.0]
blood_pressure_diastolic first 10 values: [80.0, 78.0, 90.0, 88.0, 70.0, 68.0, 95.0, 90.0, 85.0, 80.0, 65.0, 60.0, 95.0, 90.0, 85.0, 80.0, 80.0, 80.0, 98.0, 95.0]
pulse f

## 2.3-Raw Column Cleaning and Harmonization - Hospital-Related Columns

In [12]:
#Define patient-related columns
hospital_cols = ['hospitalization_required', 'hospitalization_days', 'blood_tests_required',
                 'imaging_required'
]

#Function to clean hospital-related columns
def clean_hospital_cols(df):
    #Standardize Yes/No columns (categorical)
    yes_no_cols = ['hospitalization_required', 'blood_tests_required', 'imaging_required']
    for col in yes_no_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
            # Map all variants to 'Y'/'N'
            df[col] = df[col].replace({
                'YES': 'Y', 'Y': 'Y', 'OUI': 'Y', 'N': 'N', 'NO': 'N', 'NON': 'N', '': pd.NA, 'NA': pd.NA, 'NAN': pd.NA
            })

    #Standardize hospitalization_days
    if 'hospitalization_days' in df.columns:
        # Convert to float, invalid entries become NaN
        df['hospitalization_days'] = pd.to_numeric(df['hospitalization_days'], errors='coerce')

    return df

#Apply cleaning for all datasets
for file_name, df in dfs.items():
    df = clean_hospital_cols(df)
    dfs[file_name] = df

#Quick verification
for file_name, df in dfs.items():
    print(f"=== File: {file_name} ===")
    for col in hospital_cols:
        if col in df.columns:
            print(f"{col} first 10 values: {df[col].head(10).tolist()}")
    print("\n")

=== File: Donnees_Kanza_Kambara_cleaned.csv ===
hospitalization_required first 10 values: ['Y', 'N', 'Y', 'N', 'Y', 'N', 'N', 'N', 'Y', 'N']
hospitalization_days first 10 values: [12.0, nan, 15.0, nan, 7.0, nan, nan, nan, 10.0, nan]
blood_tests_required first 10 values: ['Y', 'N', 'Y', 'N', 'Y', 'N', 'N', 'N', 'Y', 'N']
imaging_required first 10 values: ['Y', 'N', 'Y', 'N', 'N', 'N', 'N', 'N', 'Y', 'N']


=== File: Donnees_Abeni_Kambara_cleaned.csv ===
hospitalization_required first 10 values: ['N', 'N', 'Y', 'N', 'N', 'N', 'Y', 'N', 'N', 'Y']
hospitalization_days first 10 values: [nan, nan, 4.0, nan, nan, nan, 3.0, nan, nan, 5.0]
blood_tests_required first 10 values: ['N', 'N', 'Y', 'Y', 'Y', 'N', 'Y', 'N', 'Y', 'Y']
imaging_required first 10 values: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y']


=== File: Donnees_WKMP_Kambara_cleaned.csv ===
hospitalization_required first 10 values: ['N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'N', 'N']
hospitalization_days first 10 values: [nan, n

## 2.4-Merge datasets and convert to appropriate datatypes

In [13]:
#Merge all datasets into a single DataFrame
merged_df = pd.concat(dfs.values(), ignore_index=True)

#Convert categorical columns
categorical_columns = [
    'sex', 'diagnosis', 'vaccination_status',
    'hospitalization_required', 'blood_tests_required', 'imaging_required'
]

for col in categorical_columns:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].astype('category')

#Convert numeric columns
#Float columns
float_columns = ['height', 'weight', 'temperature', 'hospitalization_days']

for col in float_columns:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

#Int columns
int_columns = [
    'visit_number', 'age', 'pulse', 'oxygen_saturation',
    'blood_pressure_systolic', 'blood_pressure_diastolic'
]

for col in int_columns:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce').astype('Int64')

#Remove the 'blood_pressure' column
if 'blood_pressure' in merged_df.columns:
    merged_df = merged_df.drop(columns=['blood_pressure'])

#Convert date columns
date_columns = ['visit_date', 'birth_date']
for col in date_columns:
    if col in merged_df.columns:
        merged_df[col] = pd.to_datetime(merged_df[col], errors='coerce')

#Quick check
print(merged_df.info())
print(merged_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1097 entries, 0 to 1096
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   id_patient                1097 non-null   object        
 1   visit_number              1097 non-null   Int64         
 2   visit_date                1096 non-null   datetime64[ns]
 3   clinic_name               1097 non-null   object        
 4   patient_language          1097 non-null   object        
 5   sex                       1097 non-null   category      
 6   birth_date                1093 non-null   datetime64[ns]
 7   age                       1097 non-null   Int64         
 8   height                    1097 non-null   float64       
 9   weight                    1097 non-null   float64       
 10  temperature               1097 non-null   float64       
 11  pulse                     1095 non-null   Int64         
 12  oxygen_saturation   

## 2.5-Summary

In this notebook, we completed **Part 2 of the data cleaning pipeline**, which included:

- Standardizing and cleaning patient-related columns (IDs, names, visit numbers, demographic info.) to ensure consistency.
- Cleaning hospital-related columns (hospitalization status, days, tests, and imaging requirements) while preserving medical and numerical values.
- Merging all sources into a single dataset for further analysis.
- Converting columns to appropriate data types (numerical, categorical, datetime) for further analysis.

## 2.6-Save dataset

In [14]:
#Save merged_df to a CSV file in Kaggle
merged_df.to_csv("/kaggle/working/merged_df.csv", index = False)

print("File saved as /kaggle/working/merged_df.csv")

File saved as /kaggle/working/merged_df.csv
